# Fast embedding-blocking for source3 (GPU)

1. Runtime menu -> Change runtime type -> **T4 GPU** -> Save.
2. Run every cell below in order (Runtime -> Run all).
3. When the upload cell runs, upload `colab_source1.parquet` and `colab_source3.parquet` from your project folder (`code/business_entity_resolution/`).
4. At the end, `source3_embed_pairs.zip` downloads automatically. Unzip it into `artifacts/blocking/` in your project.

This does the same encode + nearest-neighbor search as `src/embed_block.py train source3` on your Mac, just on a GPU, and it saves each country's result to its own file as soon as it's done (so a disconnect only costs you that one country, not the whole run).

In [ ]:
!pip install -q sentence-transformers hnswlib pyarrow duckdb

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected. Go to Runtime > Change runtime type > T4 GPU, then Runtime > Restart session, and re-run from the top.")

In [ ]:
from google.colab import files
print("Upload colab_source1.parquet and colab_source3.parquet now:")
uploaded = files.upload()
assert "colab_source1.parquet" in uploaded and "colab_source3.parquet" in uploaded, \
    "Both files are required -- re-run this cell and upload both."

In [ ]:
import time, os
import duckdb
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
import torch
import hnswlib

TOP_K = 100
BATCH_SIZE = 1024  # GPU has more headroom than the Mac's 512

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, flush=True)
model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2", device=device)


def encode(names):
    return model.encode(names, batch_size=BATCH_SIZE, show_progress_bar=True,
                         normalize_embeddings=True, convert_to_numpy=True)


def safe(country):
    return "".join(c if c.isalnum() else "_" for c in country)


con = duckdb.connect()
s1_all = con.execute("select entity_id, business_name, country from read_parquet('colab_source1.parquet')").df()
o_all = con.execute("select entity_id, business_name, country from read_parquet('colab_source3.parquet')").df()

countries = sorted(set(s1_all["country"]).intersection(set(o_all["country"])))
print("countries:", countries, flush=True)

os.makedirs("out", exist_ok=True)

for country in countries:
    out_path = f"out/train_source3_{safe(country)}_embed_pairs.parquet"
    if os.path.exists(out_path):
        print(f"  {country}: already done, skipping", flush=True)
        continue

    t0 = time.time()
    s1 = s1_all[s1_all["country"] == country].reset_index(drop=True)
    o = o_all[o_all["country"] == country].reset_index(drop=True)
    if len(s1) == 0 or len(o) == 0:
        print(f"  {country}: skip (s1={len(s1)}, o={len(o)})", flush=True)
        continue

    print(f"  {country}: s1={len(s1)} o={len(o)}", flush=True)
    s1_emb = encode(s1["business_name"].fillna("").tolist())
    o_emb = encode(o["business_name"].fillna("").tolist())

    dim = o_emb.shape[1]
    k = min(TOP_K, len(o))
    index = hnswlib.Index(space="cosine", dim=dim)
    index.init_index(max_elements=len(o), ef_construction=200, M=16)
    index.add_items(o_emb, np.arange(len(o)))
    index.set_ef(max(50, k * 2))

    labels, distances = index.knn_query(s1_emb, k=k)
    del s1_emb, o_emb, index

    o_ids = o["entity_id"].to_numpy()
    s1_ids = s1["entity_id"].to_numpy()

    s1_col = np.repeat(s1_ids, k)
    o_col = o_ids[labels.reshape(-1)]
    score_col = (1 - distances).reshape(-1).astype(np.float32)

    df = pd.DataFrame({"s1_id": s1_col, "o_id": o_col, "embed_score": score_col})
    df.to_parquet(out_path, index=False)
    print(f"    done in {time.time()-t0:.1f}s, saved {out_path}: {len(df)} rows", flush=True)

print("ALL DONE", flush=True)

In [ ]:
import shutil
from google.colab import files
shutil.make_archive("source3_embed_pairs", "zip", "out")
files.download("source3_embed_pairs.zip")